# Experiment 5.2 — Frozen Local → FF/RSNN, $\tau_R$ sweep

Analysis-only notebook. Training and Slurm orchestration live under `scripts/`. The notebook never selects on test BA.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
ARTIFACT_ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_5_2_frozen_local_tauR_sweep' / 'frozen_exp3_l2_endpoint_tauR_v1'
runs = pd.read_csv(ARTIFACT_ROOT / 'runs.csv')
histories = pd.read_csv(ARTIFACT_ROOT / 'histories.csv')
local_reference = pd.read_csv(ARTIFACT_ROOT / 'local_reference.csv')
manifest = json.loads((ARTIFACT_ROOT / 'manifest.json').read_text(encoding='utf-8'))
assert len(runs) == manifest['expected_decoder_runs'] == 50
runs.head()


## 1. Validation-only summary and $\tau_R$ selection

The RSNN tau is selected by **mean validation native endpoint BA**. Test BA is only reported after that selection.


In [ ]:
summary = (
    runs.groupby(['architecture', 'shift_mem_r', 'tau_mem_r_ms'], as_index=False)
    .agg(
        val_ba_mean=('native_val_balanced_accuracy', 'mean'),
        val_ba_sd=('native_val_balanced_accuracy', 'std'),
        test_ba_mean=('native_test_balanced_accuracy', 'mean'),
        test_ba_sd=('native_test_balanced_accuracy', 'std'),
        test_macro_f1_mean=('native_test_macro_f1', 'mean'),
        hidden_rate_mean=('native_test_hidden_events_per_neuron_second', 'mean'),
        tail_fraction_mean=('native_test_hidden_tail_event_fraction', 'mean'),
        endpoint_norm_mean=('native_test_endpoint_membrane_l2_mean', 'mean'),
    )
)
rsnn_summary = summary[summary['architecture'] == 'rsnn'].copy()
best_row = rsnn_summary.sort_values(['val_ba_mean', 'tau_mem_r_ms'], ascending=[False, True]).iloc[0]
selected_rsnn_shift = int(best_row['shift_mem_r'])
selected_rsnn_tau_ms = float(best_row['tau_mem_r_ms'])
selected_rsnn = runs[(runs['architecture'] == 'rsnn') & (runs['shift_mem_r'] == selected_rsnn_shift)].copy()
print(f'Validation-selected RSNN: shift={selected_rsnn_shift}, tau={selected_rsnn_tau_ms:.1f} ms')
print(f'Selected test BA: {selected_rsnn.native_test_balanced_accuracy.mean():.4f} ± {selected_rsnn.native_test_balanced_accuracy.std():.4f}')
summary


## 2. Local information reference levels

These are computed from the exact same frozen L2 trajectory before any FF/RSNN decoder.


In [ ]:
local_ref_summary = (
    local_reference.groupby('probe_type', as_index=False)
    .agg(val_ba_mean=('val_ba', 'mean'), val_ba_sd=('val_ba', 'std'), test_ba_mean=('test_ba', 'mean'), test_ba_sd=('test_ba', 'std'))
)
local_ref_summary


In [ ]:
ref = local_ref_summary.set_index('probe_type')['test_ba_mean']
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for architecture, group in summary.groupby('architecture'):
    group = group.sort_values('tau_mem_r_ms')
    ax.plot(group['tau_mem_r_ms'], group['test_ba_mean'], marker='o', label=architecture.upper())
    ax.fill_between(
        group['tau_mem_r_ms'],
        group['test_ba_mean'] - group['test_ba_sd'],
        group['test_ba_mean'] + group['test_ba_sd'],
        alpha=0.15,
    )
for probe_type, label in [
    ('local_whole_count', 'Frozen L2 WholeCount'),
    ('local_fixed250_ordered', 'Frozen L2 Fixed250'),
    ('local_relative10_ordered', 'Frozen L2 Relative10'),
]:
    if probe_type in ref.index:
        ax.axhline(ref.loc[probe_type], linestyle='--', linewidth=1.2, label=label)
ax.set_xscale('log')
ax.set_xlabel(r'$\tau_R$ (ms)')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Frozen Local → endpoint state: FF vs RSNN')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 3. Paired recurrence contribution

For each seed and matched tau, compute `RSNN - FF`.


In [ ]:
paired = runs.pivot_table(
    index=['seed', 'shift_mem_r', 'tau_mem_r_ms'],
    columns='architecture',
    values=['native_val_balanced_accuracy', 'native_test_balanced_accuracy'],
).reset_index()
paired.columns = ['_'.join([str(x) for x in col if str(x)]) if isinstance(col, tuple) else str(col) for col in paired.columns]
paired['rsnn_minus_ff_val'] = paired['native_val_balanced_accuracy_rsnn'] - paired['native_val_balanced_accuracy_ff']
paired['rsnn_minus_ff_test'] = paired['native_test_balanced_accuracy_rsnn'] - paired['native_test_balanced_accuracy_ff']
paired_effects = (
    paired.groupby(['shift_mem_r', 'tau_mem_r_ms'], as_index=False)
    .agg(
        val_effect_mean=('rsnn_minus_ff_val', 'mean'),
        val_effect_sd=('rsnn_minus_ff_val', 'std'),
        test_effect_mean=('rsnn_minus_ff_test', 'mean'),
        test_effect_sd=('rsnn_minus_ff_test', 'std'),
        positive_test_seeds=('rsnn_minus_ff_test', lambda s: int((s > 0).sum())),
    )
)
paired_effects


In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.axhline(0.0, linewidth=1.0)
ax.errorbar(paired_effects['tau_mem_r_ms'], paired_effects['test_effect_mean'], yerr=paired_effects['test_effect_sd'], marker='o', capsize=4)
ax.set_xscale('log')
ax.set_xlabel(r'$\tau_R$ (ms)')
ax.set_ylabel('Paired test BA: RSNN - FF')
ax.set_title('Learned recurrence contribution beyond passive membrane memory')
ax.grid(alpha=0.25)
plt.show()


## 4. Hidden-state readout matrix for validation-selected conditions


In [ ]:
selected_ff_shift = int(
    summary[summary['architecture'] == 'ff'].sort_values(['val_ba_mean', 'tau_mem_r_ms'], ascending=[False, True]).iloc[0]['shift_mem_r']
)
selected_conditions = pd.concat([
    runs[(runs['architecture'] == 'ff') & (runs['shift_mem_r'] == selected_ff_shift)],
    runs[(runs['architecture'] == 'rsnn') & (runs['shift_mem_r'] == selected_rsnn_shift)],
])
probe_cols = [
    'hidden_whole_count_test_ba',
    'hidden_fixed250_ordered_test_ba',
    'hidden_relative10_ordered_test_ba',
    'hidden_uend_test_ba',
]
readout_matrix = selected_conditions.groupby(['architecture', 'shift_mem_r', 'tau_mem_r_ms'])[probe_cols].agg(['mean', 'std'])
readout_matrix


## 5. Timing recovery and Fixed250 gap

`timing_recovery = (Uend - LocalCount) / (LocalRelative10 - LocalCount)`. It is intentionally not clipped.


In [ ]:
ref_per_seed = local_reference.pivot(index='seed', columns='probe_type', values='test_ba')
diag = runs.merge(ref_per_seed, left_on='seed', right_index=True, how='left')
diag['timing_recovery'] = (
    diag['hidden_uend_test_ba'] - diag['local_whole_count']
) / (diag['local_relative10_ordered'] - diag['local_whole_count'])
diag['fixed250_gap'] = diag['local_fixed250_ordered'] - diag['hidden_uend_test_ba']
diagnostic_summary = (
    diag.groupby(['architecture', 'shift_mem_r', 'tau_mem_r_ms'], as_index=False)
    .agg(
        timing_recovery_mean=('timing_recovery', 'mean'),
        timing_recovery_sd=('timing_recovery', 'std'),
        fixed250_gap_mean=('fixed250_gap', 'mean'),
        fixed250_gap_sd=('fixed250_gap', 'std'),
    )
)
diagnostic_summary


## 6. Validation learning curves for validation-selected FF and RSNN


In [ ]:
selected_hist = histories[
    ((histories['architecture'] == 'ff') & (histories['shift_mem_r'] == selected_ff_shift))
    | ((histories['architecture'] == 'rsnn') & (histories['shift_mem_r'] == selected_rsnn_shift))
].copy()
curve = selected_hist.groupby(['architecture', 'epoch'], as_index=False).agg(
    val_ba_mean=('val_balanced_accuracy', 'mean'),
    val_ba_sd=('val_balanced_accuracy', 'std'),
)
fig, ax = plt.subplots(figsize=(8.5, 5.0))
for architecture, group in curve.groupby('architecture'):
    group = group.sort_values('epoch')
    ax.plot(group['epoch'], group['val_ba_mean'], label=architecture.upper())
    ax.fill_between(group['epoch'], group['val_ba_mean'] - group['val_ba_sd'], group['val_ba_mean'] + group['val_ba_sd'], alpha=0.15)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation endpoint BA')
ax.set_title('Validation-selected Exp5.2 learning curves')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## Interpretation checklist

- `FF(long tau) > FF(short tau)`: passive membrane persistence helps.
- `RSNN(tau) > FF(tau)`: learned recurrence contributes beyond passive memory.
- `hidden_relative10_ordered >> hidden_uend`: informative trajectory exists but endpoint consolidation remains incomplete.
- `hidden_uend ≈ local_fixed250_ordered`: recurrent endpoint state recovers most information that previously required explicit phase-addressed storage.
- Long tau with rising tail fraction: accuracy gain may be coupled to unstable autonomous persistence.
